# 02 — Data Preparation

## Objective
Define the Model 1 population, engineer structural predictors, create a stratified train/test split, and construct a leakage-safe preprocessing pipeline.

The dataset is not modified on disk. The holdout test set is reserved for a single final evaluation after model selection.

## 1. Load the dataset
The input file is expected at `data/raw/listings.csv.gz`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
if (cwd / "data" / "raw" / "listings.csv.gz").exists():
    project_root = cwd
elif (cwd.parent / "data" / "raw" / "listings.csv.gz").exists():
    project_root = cwd.parent
else:
    raise FileNotFoundError("Expected data/raw/listings.csv.gz in the project root.")

data_path = project_root / "data" / "raw" / "listings.csv.gz"
df = pd.read_csv(data_path, low_memory=False)
print("Dataset shape:", df.shape)

## 2. Define the modeling population
Only `Entire home/apt` and `Private room` are included. The upper price limit of 1,000 defines the project scope; the lower bound excludes zero or negative advertised prices.

In [ ]:
# Convert price strings such as "$130.00" to numeric values

df["price_numeric"] = pd.to_numeric(
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False),
    errors="coerce"
)

selected_types = ["Entire home/apt", "Private room"]

model_df = df.loc[
    df["room_type"].isin(selected_types)
    & df["price_numeric"].notna()
].copy()

print("Listings with valid target:", len(model_df))
print(model_df["room_type"].value_counts())
# Define the target price range for Model 1

MAX_PRICE = 1000

model_df = model_df.loc[
    (model_df["price_numeric"] > 0)
    & (model_df["price_numeric"] <= MAX_PRICE)
].copy()

print("Listings after price filtering:", len(model_df))
print("\nListings by room type:")
print(model_df["room_type"].value_counts())

print("\nTarget summary:")
print(model_df["price_numeric"].describe())

In [ ]:
# The selected sample contains only positive prices.
assert (model_df["price_numeric"] > 0).all()
print("Modeling sample:", len(model_df))

## 3. Engineer bathroom features
Bathroom counts and privacy labels are extracted from `bathrooms_text`. Missing counts are imputed later using training-only statistics.

In [ ]:
def add_bathroom_features(data):
    """
    Extract bathroom count and privacy type
    from the bathrooms_text column.
    """
    result = data.copy()

    text = result["bathrooms_text"].astype("string")

    numbers = pd.to_numeric(
        text.str.extract(
            r"(\d+(?:\.\d+)?)",
            expand=False
        ),
        errors="coerce"
    )

    half_bath = (
        text.str.contains("half-bath", case=False, na=False)
        & numbers.isna()
    )

    result["bathrooms_count"] = numbers.mask(half_bath, 0.5)

    text_lower = text.str.lower()

    result["bathroom_type"] = "unspecified"

    result.loc[
        text_lower.str.contains("private", na=False),
        "bathroom_type"
    ] = "private"

    result.loc[
        text_lower.str.contains("shared", na=False),
        "bathroom_type"
    ] = "shared"

    result.loc[
        text_lower.isna(),
        "bathroom_type"
    ] = "missing"

    return result

In [ ]:
model_df = add_bathroom_features(model_df)
print("Missing bathroom counts:", model_df["bathrooms_count"].isna().sum())
display(model_df["bathroom_type"].value_counts())

## 4. Select features and reserve the test set
The split is stratified by `room_type` to preserve the proportion of entire homes and private rooms. The random seed is fixed for reproducibility.

In [ ]:
from sklearn.model_selection import train_test_split

numeric_features = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms_count"
]

categorical_features = [
    "room_type",
    "property_type",
    "bathroom_type"
]

feature_columns = numeric_features + categorical_features

X = model_df[feature_columns].copy()
y = model_df["price_numeric"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=X["room_type"]
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

print("\nTraining room types:")
print(X_train["room_type"].value_counts())

print("\nTest room types:")
print(X_test["room_type"].value_counts())

print("\nMissing beds in training:")
print(X_train.groupby("room_type")["beds"].apply(
    lambda s: s.isna().sum()
))

## 5. Missing beds: group-specific medians
The number of beds is imputed using the median within each room type. A binary `beds_missing` indicator preserves information about missingness. The medians are learned in `fit`, never from validation or test rows.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class BedsImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.medians_ = X.groupby("room_type")["beds"].median()
        self.global_median_ = X["beds"].median()
        return self

    def transform(self, X):
        X = X.copy()

        X["beds_missing"] = X["beds"].isna().astype(int)

        group_medians = X["room_type"].map(self.medians_)

        X["beds"] = (
            X["beds"]
            .fillna(group_medians)
            .fillna(self.global_median_)
        )

        return X

## 6. Rare property types
Property types appearing fewer than 30 times in a training fold are grouped into `Other`. Unseen categories are also assigned to `Other`.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class RareCategoryGrouper(BaseEstimator, TransformerMixin):

    def __init__(self, column="property_type", min_frequency=30):
        self.column = column
        self.min_frequency = min_frequency

    def fit(self, X, y=None):
        counts = X[self.column].value_counts()

        self.frequent_categories_ = set(
            counts[counts >= self.min_frequency].index
        )

        return self

    def transform(self, X):
        X = X.copy()

        X[self.column] = (
            X[self.column]
            .where(
                X[self.column].isin(self.frequent_categories_),
                "Other"
            )
        )

        return X

## 7. Assemble preprocessing
Numeric variables use median imputation. Categorical variables are one-hot encoded. `BedsImputer` runs first because it creates `beds_missing`.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Numeric features
numeric_features = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms_count",
    "beds_missing"
]

# Categorical features
categorical_features = [
    "room_type",
    "property_type",
    "bathroom_type"
]

# Numeric preprocessing
numeric_transformer = SimpleImputer(
    strategy="median"
)

# Categorical preprocessing
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

# Combine numeric and categorical transformations
column_transformer = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

# Complete preprocessing pipeline
preprocessing_pipeline = Pipeline(
    steps=[
        ("beds_imputer", BedsImputer()),
        ("rare_categories", RareCategoryGrouper(
            column="property_type",
            min_frequency=30
        )),
        ("column_transformer", column_transformer)
    ]
)

## 8. Verify the output on the training set

The preprocessing pipeline is fitted to the training set here **only to inspect its output**: the transformed shape, missing or infinite values, and generated feature names. This check does not train or select a predictive model.

During cross-validation in the next notebook, preprocessing will be included inside each model pipeline. Each fold will therefore learn its own imputation medians and frequent property types from that fold's training portion only. The preprocessed matrix created below must **not** be reused as input to cross-validation, since that would allow information from validation folds to influence preprocessing.


In [ ]:
# Fit and transform training data only
X_train_processed = preprocessing_pipeline.fit_transform(X_train)

print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

# Check for missing values in the processed matrix
import numpy as np

if hasattr(X_train_processed, "toarray"):
    processed_values = X_train_processed.toarray()
else:
    processed_values = X_train_processed

print("Missing values after preprocessing:",
      np.isnan(processed_values).sum())

print("Infinite values:",
      np.isinf(processed_values).sum())

In [ ]:
# Inspect the features generated by the preprocessing pipeline

feature_names = (
    preprocessing_pipeline
    .named_steps["column_transformer"]
    .get_feature_names_out()
)

print("Number of generated features:", len(feature_names))

for name in feature_names:
    print(name)

assert len(feature_names) == X_train_processed.shape[1]
assert any("beds_missing" in name for name in feature_names)
assert any("property_type_Other" in name for name in feature_names)

print("\nFeature checks passed.")

## Preparation summary

The original predictors comprise seven structural features. The preprocessing pipeline expands them into numeric model inputs. The test set is not used to learn imputations, frequent categories, or model parameters.

The next notebook evaluates a median baseline, Ridge regression, and histogram gradient boosting using five-fold cross-validation, then evaluates the selected model on the holdout test set.